# Metaclasses and Class Creation

A **metaclass** is the class of a class — it controls how classes are created.

```
object ← instance of → class ← instance of → metaclass
```

**Most of the time you don't need metaclasses.** But they're powerful for:
- ORMs (like Django's Model)
- Plugin systems
- API validation
- Enforcing coding standards

Python's default metaclass is `type`.

## 1. `type()` — The Default Metaclass

`type(name, bases, namespace)` creates a new class dynamically.

In [ ]:
# Check the type (metaclass) of various objects
print(type(42))       # <class 'int'>
print(type('hello'))  # <class 'str'>
print(type([]))       # <class 'list'>

class Dog:
    pass

d = Dog()
print(type(d))    # <class '__main__.Dog'>
print(type(Dog))  # <class 'type'>  ← Dog's metaclass is 'type'
print(type(type)) # <class 'type'>  ← type is its own metaclass

In [ ]:
# Create a class dynamically with type()
# type(name, bases, dict)
Animal = type('Animal', (object,), {
    'sound': 'generic sound',
    'speak': lambda self: print(f'{self.__class__.__name__} says: {self.sound}')
})

Dog = type('Dog', (Animal,), {'sound': 'Woof'})
Cat = type('Cat', (Animal,), {'sound': 'Meow'})

d = Dog()
c = Cat()
d.speak()
c.speak()
print(isinstance(d, Animal))  # True

## 2. How Class Creation Works

When Python processes a `class` statement:
1. Calls `metaclass.__prepare__()` to create the namespace dict
2. Executes the class body in that namespace
3. Calls `metaclass(name, bases, namespace)` to create the class object

In [ ]:
class MyMeta(type):
    def __new__(mcs, name, bases, namespace):
        print(f'Creating class: {name}')
        print(f'  Bases: {bases}')
        print(f'  Attributes: {[k for k in namespace if not k.startswith("_")]}')
        cls = super().__new__(mcs, name, bases, namespace)
        return cls

class Animal(metaclass=MyMeta):
    sound = 'generic'
    
    def speak(self):
        print(self.sound)

class Dog(Animal):  # Dog also uses MyMeta (inherited)
    sound = 'Woof'

## 3. Practical: Singleton Pattern via Metaclass

In [ ]:
class SingletonMeta(type):
    """Metaclass that ensures only one instance per class."""
    _instances = {}
    
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]


class DatabaseConnection(metaclass=SingletonMeta):
    def __init__(self, url='localhost'):
        self.url = url
        print(f'Connecting to {url}')


# Only one instance ever created
db1 = DatabaseConnection('db1.example.com')
db2 = DatabaseConnection('db2.example.com')  # second call — same instance!

print(db1 is db2)     # True — same object
print(db1.url)         # 'db1.example.com' — original value

## 4. Practical: Attribute Validation Metaclass

In [ ]:
class ValidateMeta(type):
    """Metaclass that validates method signatures."""
    
    def __new__(mcs, name, bases, namespace):
        # Ensure all public methods have docstrings
        for key, value in namespace.items():
            if callable(value) and not key.startswith('_'):
                if not value.__doc__:
                    raise TypeError(
                        f'{name}.{key}() is missing a docstring'
                    )
        return super().__new__(mcs, name, bases, namespace)


# This works — all methods have docstrings
class GoodAPI(metaclass=ValidateMeta):
    def create(self, data):
        """Create a new resource."""
        pass
    
    def delete(self, id):
        """Delete resource by id."""
        pass

print('GoodAPI created successfully')

# This fails — missing docstring
try:
    class BadAPI(metaclass=ValidateMeta):
        def create(self, data):
            pass  # no docstring!
except TypeError as e:
    print(f'Error: {e}')

## 5. Class Decorators as a Simpler Alternative

For many metaclass use cases, a **class decorator** is simpler:

In [ ]:
def singleton(cls):
    """Decorator to make a class a singleton."""
    instances = {}
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    return get_instance


@singleton
class Config:
    def __init__(self, env='development'):
        self.env = env
        print(f'Config created for {env}')


c1 = Config('production')
c2 = Config('development')  # same instance!
print(c1 is c2)   # True
print(c1.env)     # 'production'

## Mini-Project: Auto-Registering Plugin System

In [ ]:
class PluginMeta(type):
    """Metaclass that automatically registers all subclasses."""
    registry = {}
    
    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)
        # Don't register the base class itself
        if bases:  # has at least one parent
            plugin_name = namespace.get('name', name.lower())
            mcs.registry[plugin_name] = cls
            print(f'Registered plugin: {plugin_name}')
        return cls


class Exporter(metaclass=PluginMeta):
    """Base class for all exporters."""
    name = None  # subclasses set this
    
    def export(self, data):
        raise NotImplementedError


class CSVExporter(Exporter):
    name = 'csv'
    
    def export(self, data):
        lines = ['name,value']
        for item in data:
            lines.append(f"{item['name']},{item['value']}")
        return '\n'.join(lines)


class JSONExporter(Exporter):
    name = 'json'
    
    def export(self, data):
        import json
        return json.dumps(data, indent=2)


class MarkdownExporter(Exporter):
    name = 'markdown'
    
    def export(self, data):
        lines = ['| Name | Value |', '|------|-------|']
        for item in data:
            lines.append(f"| {item['name']} | {item['value']} |")
        return '\n'.join(lines)


# Use the registry to get any exporter by name
def export_data(format_name, data):
    registry = PluginMeta.registry
    if format_name not in registry:
        raise ValueError(f'Unknown format: {format_name}. Available: {list(registry)}')
    exporter = registry[format_name]()
    return exporter.export(data)


# Test data
data = [{'name': 'Alice', 'value': 95}, {'name': 'Bob', 'value': 82}]

print('\n=== CSV ===')
print(export_data('csv', data))

print('\n=== JSON ===')
print(export_data('json', data))

print('\n=== Markdown ===')
print(export_data('markdown', data))

print('\n=== Registry ===')
print('Registered plugins:', list(PluginMeta.registry.keys()))